# Stage II (lesions) - 04. Lesion masks for the Stage I-routed images

In the cascade, Stage II runs only on the images that pass the Stage I gate. This notebook
writes one binary mask per image and lesion type, in the form that LG-DRG (Stage III) reads.

1. **Configuration** - input folder(s) and output folders.
2. **Mask generation** - for each lesion model (deployment pipeline of the original
   notebooks): MA and HE crop the retina with a minimum-enclosing-circle square, resize to
   1024 x 1024 and run one forward pass; EX and CWS crop the retinal rectangle, letterbox to
   1024 x 1024 and average identity, horizontal- and vertical-flip probabilities. The
   probability map is resized back to the crop, thresholded (MA 0.01, HE 0.20, EX 0.325,
   CWS 0.40) and pasted into an empty mask of the original image size. CWS masks are then
   closed (5 x 5) and cleaned of components smaller than 40 pixels (scaled by the crop area).
3. **Check** - every input image has a mask that LG-DRG can resolve.

**Naming expected by LG-DRG.** LG-DRG resolves the mask of image `<stem>.<ext>` with
`_resolve(stem, mask_dirs[lesion], exts, mask_suffix[lesion])`: it looks for
`<mask_dir>/<stem><suffix>`, then `<stem><suffix>` + one of
`.png, .jpg, .ppm, .jpeg, .tif, .tiff`, then a glob `<stem><suffix>.*`. The suffixes are

| lesion | `mask_suffix` | file written here | output folder (`outputs/stage2_masks/` / `data/stage3/dev_masks/`) |
|---|---|---|---|
| MA | `_mask` | `<stem>_mask.png` | `test/ma` / `ma` |
| HE | `_he_mask` | `<stem>_he_mask.png` | `test/he` / `he` |
| EX | `_ex_mask` | `<stem>_ex_mask.png` | `test/ex` / `ex` |
| CWS | `_cws_mask` | `<stem>_cws_mask.png` | `test/cws` / `cws` |

Each lesion needs its own folder (MA and vessel masks share the suffix `_mask`). Masks are
uint8 PNGs (0/255) of the original image size, because LG-DRG crops them with the fundus box
computed on the image and binarises them at 127.

## Step 1. Configuration

In [ ]:
from pathlib import Path

LESIONS = ["MA", "HE", "EX", "CWS"]
CHECKPOINT_ROOT = Path("checkpoints/stage2/hsmoe_aunet")  # <LESION>/best_model.pth

# Input folder -> output root. Masks are written to <output root>/<lesion>/<stem><suffix>.png.
#   "test": test images routed to Stage II by the Stage I gate (notebook stage1_screening/03)
#   "dev" : DR-positive development images, whose masks are LG-DRG's training input
JOBS = {
    "test": (Path("outputs/stage1_gate/dr_from_test"), Path("outputs/stage2_masks/test")),
    "dev": (Path("data/stage3/dev_images"), Path("data/stage3/dev_masks")),
}
OUTPUT_FOLDER = "{lesion}"   # e.g. outputs/stage2_masks/test/ma
BATCH_SIZE = 4

In [ ]:
import gc

import torch

from hierarchiretina.stage2.hsmoe_aunet import get_lesion_config, load_hsmoe_aunet
from hierarchiretina.stage2.lesion_inference import segment_folder

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Step 2. Mask generation

In [ ]:
reports = []
for lesion in LESIONS:
    model = load_hsmoe_aunet(lesion, CHECKPOINT_ROOT / lesion / "best_model.pth",
                             map_location=DEVICE).to(DEVICE).eval()
    for job, (in_dir, out_root) in JOBS.items():
        out_dir = out_root / OUTPUT_FOLDER.format(job=job, lesion=lesion.lower())
        rep = segment_folder(model, lesion, in_dir, out_dir, device=DEVICE,
                             batch_size=BATCH_SIZE, use_amp=DEVICE.type == "cuda")
        reports.append({"job": job, **rep})
        print(f"{lesion} [{job}]: {rep['saved']}/{rep['n_images']} masks at threshold "
              f"{rep['threshold']} -> {out_dir}")
    del model
    gc.collect(); torch.cuda.empty_cache()

## Step 3. Check that LG-DRG can resolve every mask

In [ ]:
missing = {}
for lesion in LESIONS:
    cfg = get_lesion_config(lesion)
    for job, (in_dir, out_root) in JOBS.items():
        out_dir = out_root / OUTPUT_FOLDER.format(job=job, lesion=lesion.lower())
        stems = [p.stem for p in in_dir.iterdir() if p.suffix.lower() in cfg.inference_exts]
        miss = [s for s in stems if not (out_dir / f"{s}{cfg.mask_suffix}.png").exists()]
        missing[(lesion, job)] = len(miss)
missing